# M4 実行報告・参照報告

最終更新: 2026-07-20 UTC

## 結論

| milestone | run ID | 判定 | checkpoint | certification |
|---|---|---|---:|---|
| M3 | `M3-20260720T013551Z-ae995e91e861` | 独立レビュー・受理済み | 14 | `NOT_CERTIFIED` |
| M4 | `M4-20260720T021737Z-b9c9514fed11` | `M4_COMPLETE / BLOCKED_MATH` | 14 | `NOT_CERTIFIED` |
| M5 | 未開始 | **進行不可** | - | - |

M4 の forward derivative 実装、error ledger、CPU/GPU test、checkpoint/restart gate は全18項目 PASS しました。一方、決定論的誤差境界が8項目不足しており、one-step enclosure は作れません。したがって適切な判定は **M4実装は完了、数学的には `BLOCKED_MATH`、M5へは進まない** です。`CERTIFIED` と判定してはいけません。


## M4 実測結果

- backend: `torch_cuda_opt_einsum`、NVIDIA RTX A4000、CUDA FP64、TF32 disabled
- symmetry-reduced source: 5 channels (`temporal_link`, `spatial_link`, `electric_like`, `magnetic_like`, `low_representation`)
- source symmetry residual / zero-source tangent: ともに厳密な配列比較で 0
- projected/coarse tensor: `16 x 16`、product rule と normalization derivative を全項実装
- normalization scale: `1.02887978932229`、normalized primal norm: `0.9999999999999999`
- normalized tangent norms: temporal `0.41433664`、spatial `0.44414733`、electric `0.67703798`、magnetic `0.20395204`、low-representation `0.12126689`
- centered finite-difference: 全5 channel で step 半減時に収束、最大 final relative error `1.4711130563690561e-07`
- finite difference は regression のみで、proof bound ではない
- error DAG: 12 terms、9 leaves、double-counting check PASS
- rigorous leaf: 1、heuristic leaf: 2、missing leaf: 6
- primal partial estimate: `0.3441048053215534`、tangent partial estimate: `0.344104904921495`。いずれも enclosure ではない
- CPU tests: 66 passed、6 deselected
- required GPU tests: 6 passed、66 deselected
- fresh-process resume / derivative checkpoint restore / corrupt-newest fallback: PASS
- peak CPU memory: 518,832 KiB（約506.7 MiB）
- peak GPU allocated memory: 33,568,768 bytes（約32.0 MiB）
- checkpoint: 65,970 bytes、save 0.215 s、verify 0.00269 s
- 独立再読込: `ckpt_000014`、queue 6/6 done、破損候補スキップ0、完了済みのため再計算なし


## Error ledger と人間の判定基準

| error term | 区分 | deterministic upper bound | 判定 |
|---|---|---:|---|
| M2 dense-armillary basis equivalence | rigorous | 0 | 使用可 |
| M3 RSVD projection residual | heuristic | なし | enclosure に使用不可 |
| fixed-basis variation residual | heuristic | なし | enclosure に使用不可 |
| forward tangent regression residual | heuristic | なし | regression のみ |
| initial representation tail | missing | なし | **block** |
| input radius propagation | missing | なし | **block** |
| GPU rounding and backward error | missing | なし | **block** |
| omitted fusion and channel tail | missing | なし | **block** |
| normalization and denominator error | missing | なし | **block** |
| cutoff and rank dependence | missing | なし | **block** |
| primal output partial radius | missing aggregate | なし | enclosure ではない |
| tangent output partial radius | missing aggregate | なし | enclosure ではない |

report の missing-bound 一覧には、上の6 missing leaf に加えて deterministic bound のない RSVD residual と tangent regression residual も含まれ、合計8項目です。固定基底の変分は RSVD branch の子として明示し、二重加算していません。有限・非負の partial estimate が存在しても、欠落値を0と見なしてはいけません。


## 参照先

- M4 report: `/storage/validated_4d_su2_rg/runs/M4-20260720T021737Z-b9c9514fed11/reports/M4_report.json`
- M4 acceptance: `/storage/validated_4d_su2_rg/runs/M4-20260720T021737Z-b9c9514fed11/reports/M4_acceptance.json`
- M4 manifest: `/storage/validated_4d_su2_rg/runs/M4-20260720T021737Z-b9c9514fed11/run_manifest.json`
- M4 checkpoint: `/storage/validated_4d_su2_rg/runs/M4-20260720T021737Z-b9c9514fed11/checkpoints/ckpt_000014`
- M4 初回実行 notebook（ハッシュ固定済み）: `notebooks/50_m4_derivatives.ipynb`
- M4 人手再開 notebook: `notebooks/51_m4_resume_existing.ipynb`
- M3 判定 notebook: `notebooks/45_m3_execution_reference_report.ipynb`
- M3→M4 受理記録: `audit/m3_accepted_parent.json`
- 実行ガイド: `README.md`


In [ ]:
from pathlib import Path
import hashlib
import json

RUN_ROOT = Path('/storage/validated_4d_su2_rg/runs/M4-20260720T021737Z-b9c9514fed11')
paths = {
    'M4 report': RUN_ROOT / 'reports/M4_report.json',
    'M4 acceptance': RUN_ROOT / 'reports/M4_acceptance.json',
    'M4 manifest': RUN_ROOT / 'run_manifest.json',
    'M4 checkpoint hashes': RUN_ROOT / 'checkpoints/ckpt_000014/hashes.json',
}
expected = {
    'M4 report': '7762ae9e320843956ef61cf16b2b1cc0ab34b5c2e5b6ee306870fcd451a6a0a6',
    'M4 acceptance': 'df6db25937a5524264213448ab0270a6d4be12dbcc89b35ca392f022bf94c163',
    'M4 manifest': '569b3b53df6e560076e00cae33864f91fee653f884b69b48289c6bac84a82e16',
    'M4 checkpoint hashes': '00af079071205e916c5b2faaab489d0b18cabd0d1a1d53344a5b15f514f1946f',
}
checks = {}
for label, path in paths.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None
    checks[label] = {'path': str(path), 'sha256': digest, 'PASS': digest == expected[label]}
if not all(item['PASS'] for item in checks.values()):
    raise RuntimeError('M4参照artifactが欠落または変更されています。M5を開始できません。')

report = json.loads(paths['M4 report'].read_text(encoding='utf-8'))
acceptance = json.loads(paths['M4 acceptance'].read_text(encoding='utf-8'))
manifest = json.loads(paths['M4 manifest'].read_text(encoding='utf-8'))
ledger = report['results']['M4_ERROR_LEDGER']['result']['ledger']
summary = ledger['summary']
required = {
    'report phase': report.get('phase') == 'M4_COMPLETE',
    'blocked math': report.get('milestone_status') == 'BLOCKED_MATH',
    'enclosure blocked': report.get('enclosure_status') == 'BLOCKED_MATH',
    'report not certified': report.get('certification_status') == 'NOT_CERTIFIED',
    'all 18 report gates': len(report.get('acceptance_gates', {})) == 18 and all(report['acceptance_gates'].values()),
    'acceptance pass': acceptance.get('status') == 'PASS',
    'acceptance blocked': acceptance.get('enclosure_status') == 'BLOCKED_MATH',
    'ledger not enclosure ready': summary.get('enclosure_ready') is False,
    'ledger term count': summary.get('term_count') == 12,
    'only one rigorous leaf': summary.get('rigorous_leaf_count') == 1,
    'missing bounds are explicit': len(report.get('missing_bound_terms', [])) == 8,
    'initial session hard return': report['config'].get('initial_hard_return_s') == 7200,
    'resumed session hard return': report['config'].get('hard_return_s') == 19800,
    'checkpoint interval': report['config'].get('checkpoint_interval_s') == 900,
}
if not all(required.values()):
    raise RuntimeError(f'M4 fail-closed check failed: {required}')

print(json.dumps({
    '判定': 'M4_COMPLETE / BLOCKED_MATH / NOT_CERTIFIED。M5へ進行不可。',
    'artifact_checks': checks,
    'required_checks': required,
    'acceptance_gate_count': len(report['acceptance_gates']),
    'error_ledger_summary': summary,
    'missing_bound_terms': report['missing_bound_terms'],
    'heuristic_error_terms': report['heuristic_error_terms'],
    'rigorous_error_terms': report['rigorous_error_terms'],
}, ensure_ascii=False, indent=2))


## 人間による再開手順

同じ M4 run を別セッションから再確認または再開する場合、fresh kernel で `notebooks/51_m4_resume_existing.ipynb` を上から実行します。この再開Notebookは、下記と同じ環境変数を承認済み永続markerの検証後にkernel内へ設定します。

```bash
export VALIDATED_RG_PERSIST_ROOT=/storage/validated_4d_su2_rg
export VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT
export VALIDATED_RG_M3_RUN_ID=M3-20260720T013551Z-ae995e91e861
export VALIDATED_RG_M4_RUN_ID=M4-20260720T021737Z-b9c9514fed11
```

orchestrator は最新の valid checkpoint を SHA-256 検証してロードします。最新が破損していれば直前の valid checkpoint へ戻し、`running` item は `pending` に戻します。保存は15分周期と phase/work-item 境界です。

今回だけの初回セッションは、1時間30分以降に長い仕事を始めず、1時間45分で drain、1時間50分で final save、2時間で必ず戻る設定でした。**次回以降の再開セッション**は通常設定に戻り、5時間以降に長い仕事を始めず、5時間15分で drain、5時間20分で final save、5時間30分で必ず戻ります。現在の run はすでに `M4_COMPLETE` なので、再開時は保存済み結果を検証して表示し、再計算しません。
